# 01 · Retail EDA (Exploratory Data Analysis)
**Goal:** understand the business before forecasting anything.

We answer 5 simple questions:
1. Is the data trustworthy?
2. How is revenue trending?
3. Is there seasonality?
4. Which products matter most?
5. Which countries buy the most?

> Run `python run.py` once before this notebook.

In [ ]:
import sys, json
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from src import config

plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25})
clean = pd.read_csv(config.DATA_PROCESSED / 'clean_transactions.csv', parse_dates=['InvoiceDate'])
clean.head()

## 1 · Is the data trustworthy?
The cleaning step records how many rows each rule removed.

In [ ]:
report = json.loads((config.DATA_PROCESSED / 'cleaning_report.json').read_text())
pd.Series(report, name='rows').to_frame()

In [ ]:
print('Missing values per column:')
clean.isna().sum()

*Missing `CustomerID` is normal (guest checkouts). We forecast **product demand**, so we don't need it.*

## 2 · Revenue trend

In [ ]:
monthly = clean.groupby(clean['InvoiceDate'].dt.to_period('M').astype(str))['Revenue'].sum()
ax = monthly.plot(kind='bar', color='#4F46E5')
ax.set_title('Monthly revenue', loc='left', fontweight='bold'); ax.set_xlabel('')
plt.show()

## 3 · Seasonality
Which weekday sells the most? Demand patterns like this decide when to restock.

In [ ]:
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_day = clean.groupby(clean['InvoiceDate'].dt.day_name())['Revenue'].sum().reindex(order).dropna()
ax = by_day.plot(kind='bar', color='#F97316'); ax.set_title('Revenue by weekday', loc='left', fontweight='bold'); ax.set_xlabel('')
plt.show()

In [ ]:
weekly = pd.read_csv(config.DATA_PROCESSED / 'weekly_demand.csv', parse_dates=['Week'])
total = weekly.groupby('Week')['Units'].sum()
ax = total.plot(color='#4F46E5', lw=2)
ax.set_title('Weekly units sold (top products)', loc='left', fontweight='bold'); ax.set_xlabel('')
plt.show()

## 4 · Product analysis
The **80/20 rule**: a small share of products usually creates most of the revenue.

In [ ]:
prod = clean.groupby('Description').agg(Units=('Quantity','sum'), Revenue=('Revenue','sum')).sort_values('Revenue', ascending=False)
top10 = prod.head(10).iloc[::-1]
ax = top10['Revenue'].plot(kind='barh', color='#10B981')
ax.set_title('Top 10 products by revenue', loc='left', fontweight='bold'); ax.set_ylabel('')
plt.show()

In [ ]:
share = prod['Revenue'].cumsum() / prod['Revenue'].sum()
n20 = int(len(prod) * 0.2)
print(f'Top 20% of products ({n20} items) generate {share.iloc[n20-1]:.0%} of revenue')

## 5 · Country analysis

In [ ]:
country = clean.groupby('Country')['Revenue'].sum().sort_values(ascending=False)
print(f'Top country share: {country.iloc[0]/country.sum():.0%}  ({country.index[0]})')
ax = country.iloc[1:9].iloc[::-1].plot(kind='barh', color='#4F46E5')
ax.set_title('Top export countries (excluding #1)', loc='left', fontweight='bold'); ax.set_ylabel('')
plt.show()

## ✍️ Key takeaways (write yours here!)
Clients love a short summary. Fill in your own findings:
- **Trend:** …
- **Seasonality:** …
- **Products:** …
- **Countries:** …
- **Next step:** forecast demand for the best sellers → `02_forecasting_experiments.ipynb`